# gnomAD v4 VCF to CSV Conversion Pipeline Demo

This notebook demonstrates the exact data processing pipeline used in `GeneratingCSV.py` to convert gnomAD v4 VCF files into analysis-ready CSV format for Hardy-Weinberg Equilibrium studies.

## Pipeline Overview

1. Load reference datasets (AGAIN, BPHunter, HGMD, unmappable regions)
2. **VCFtoCSV_Exome()** - Convert VCF to CSV with full annotations
3. **Filter_filter()** - Apply initial quality filters
4. **csvEachPop_Exome()** - Create population-specific CSV files
5. **Population-specific filtering** - Apply AC, call rate, and additional filters

**Sample**: First 100,000 variants from chromosome 21 for demonstration

In [1]:
# Import required packages
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
import sys
sys.path.insert(0, '../scripts')  # locate pipeline modules
import pandas as pd
import csv
from datetime import datetime
import math
import numpy as np
import os.path
from os import path
import matplotlib.pyplot as plt

from ParsingFiltering_ExomeV4 import VCFtoCSV_Exome, Filter_filter, csvEachPop_Exome, Filter_BI
from ParsingFiltering_ExomeV4 import Filter_Auto_AC_CallRate, add_MORE_Exome

## 1. Define Parameters (Exact Copy from GeneratingCSV.py)

In [2]:
# Copy exact parameters from GeneratingCSV.py
medList= ['']
quantityList= ['AC', 'AN', 'AF','nhomalt']
ethnList = ['total','afr', 'amr', 'nfe', 'sas', 'fin', 'asj', 'eas']
sexList=['']

# Generate p-value list (exact copy)
p_valueList=[]
for i in ethnList:
    for j in medList:
        if j=='':
            p_valueList.append('pv_'+i)
        else:
            p_valueList.append(j+'_pv_'+i)

# Sample sizes (exact copy)
nbAllesPerEthn=[729178*2,16677*2,22359*2,554421*2,43107*2,26704*2,13057*2,19834*2]

print(f"Populations: {ethnList}")
print(f"Quantities: {quantityList}")
print(f"HWE P-value fields: {p_valueList}")
print(f"Sample sizes: {[n//2 for n in nbAllesPerEthn]} individuals")

Populations: ['total', 'afr', 'amr', 'nfe', 'sas', 'fin', 'asj', 'eas']
Quantities: ['AC', 'AN', 'AF', 'nhomalt']
HWE P-value fields: ['pv_total', 'pv_afr', 'pv_amr', 'pv_nfe', 'pv_sas', 'pv_fin', 'pv_asj', 'pv_eas']
Sample sizes: [729178, 16677, 22359, 554421, 43107, 26704, 13057, 19834] individuals


## 2. Load Reference Datasets (Exact Copy from GeneratingCSV.py)

### ⚠️ Important Note on External Database Licensing

**HGMD Data Confidentiality**: The Human Gene Mutation Database (HGMD) contains proprietary data that requires licensing from Cardiff University. **HGMD data files are NOT included in this public repository** due to licensing restrictions.

- **For demonstration**: The pipeline will run with empty HGMD annotations (default value = 0)
- **For your own research**: Obtain HGMD license at https://www.hgmd.cf.ac.uk/
- **Pipeline compatibility**: The code handles missing HGMD files gracefully

In [3]:
# Load AGAIN high-risk variants (exact copy from GeneratingCSV.py)
listAGAIN=[]
try:
    csvFile= open('../data/pipeline/gnomad_genomes_v3_grch38_AGAIN_highrisk.txt','r')
    csvLines= csvFile.readlines()
    for i in range(1,len(csvLines)):
        info = csvLines[i][:-1].split("\t")
        listAGAIN.append([info[0],info[1],info[3],info[4]])
    csvFile.close()
    print(f"Loaded {len(listAGAIN)} AGAIN variants")
except FileNotFoundError:
    print("AGAIN file not found - proceeding without AGAIN annotations")

# Load BPHunter high-risk variants (exact copy)
listBPHUNTER=[]
try:
    csvFile= open('../data/pipeline/gnomad_genomes_v3_grch38_BPHunter_highrisk.txt','r')
    csvLines= csvFile.readlines()
    for i in range(1,len(csvLines)):
        info = csvLines[i][:-1].split("\t")
        listBPHUNTER.append([info[0],info[1],info[3],info[4]])
    csvFile.close()
    print(f"Loaded {len(listBPHUNTER)} BPHunter variants")
except FileNotFoundError:
    print("BPHunter file not found - proceeding without BPHunter annotations")

# Load HGMD variants - CONFIDENTIAL DATA NOT INCLUDED
print("⚠️  HGMD (Human Gene Mutation Database) - CONFIDENTIAL DATA")
print("   HGMD data requires licensing and is NOT included in this public repository.")
print("   To use HGMD annotations in your own analysis:")
print("   1. Visit https://www.hgmd.cf.ac.uk/")
print("   2. Obtain appropriate academic/commercial license")
print("   3. Download mutation data files (allmut_hg38_vcf_perChr_*.tsv)")
print("   4. Place them in the same directory as this notebook")
print("   5. Re-run this cell to load the data")
print()

listHGMD=[]
hgmd_files_found = 0
for i in range(1,23):
    try:
        csvFile= open('allmut_hg38_vcf_perChr_'+str(i)+'.tsv','r')
        csvLines= csvFile.readlines()
        listVariants = []
        for line in csvLines[1:len(csvLines)]:
            info = line[:-1].split("\t")
            listVariants.append([str('chr'+info[16]),str(info[17]),info[46],info[47],info[23]])
        listHGMD.append(listVariants)
        csvFile.close()
        hgmd_files_found += 1
    except FileNotFoundError:
        listHGMD.append([])  # Empty list for missing chromosome

total_hgmd = sum(len(chr_vars) for chr_vars in listHGMD)
if total_hgmd > 0:
    print(f"✅ Loaded {total_hgmd} HGMD variants across {hgmd_files_found} chromosomes")
else:
    print(f"ℹ️  No HGMD files found - HGMD annotations will be set to default value (0)")
    print(f"   Pipeline will continue normally with empty HGMD annotations")

# pLOF consequences (exact copy)
listpLOF=['splice_acceptor_variant','splice_donor_variant','stop_gained',
          'frameshift_variant','start_lost']


print(f"\nDefined {len(listpLOF)} pLOF consequences")

Loaded 14707 AGAIN variants
Loaded 12775 BPHunter variants
⚠️  HGMD (Human Gene Mutation Database) - CONFIDENTIAL DATA
   HGMD data requires licensing and is NOT included in this public repository.
   To use HGMD annotations in your own analysis:
   1. Visit https://www.hgmd.cf.ac.uk/
   2. Obtain appropriate academic/commercial license
   3. Download mutation data files (allmut_hg38_vcf_perChr_*.tsv)
   4. Place them in the same directory as this notebook
   5. Re-run this cell to load the data

ℹ️  No HGMD files found - HGMD annotations will be set to default value (0)
   Pipeline will continue normally with empty HGMD annotations

Defined 5 pLOF consequences


In [4]:
# Generate FST header (exact copy)
FSTs_header = []
for i in range(len(ethnList)):
    for j in range(i+1,len(ethnList)):        
        if i == 0:
            FSTs_header.append('FST_other'+'_vs_'+ethnList[j])
        else:
            FSTs_header.append('FST_'+ethnList[i]+'_vs_'+ethnList[j])

print(f"Generated {len(FSTs_header)} FST comparison fields")
print("FST:", FSTs_header)

# Load unmappable intervals (exact copy)
Merged_Intervals_37 = []
csvFile= open('../data/pipeline/UM_37_Merged_Intervals.csv','r')
csvLines= csvFile.readlines()
for line in csvLines[0:len(csvLines)]:
    info = line[:-1].split(",")
    intervals = []
    i=0
    while i < len(info):
        intervals.append([int(info[i]),int(info[i+1])])
        i+=2
    Merged_Intervals_37.append(intervals)
csvFile.close()


Generated 28 FST comparison fields
FST: ['FST_other_vs_afr', 'FST_other_vs_amr', 'FST_other_vs_nfe', 'FST_other_vs_sas', 'FST_other_vs_fin', 'FST_other_vs_asj', 'FST_other_vs_eas', 'FST_afr_vs_amr', 'FST_afr_vs_nfe', 'FST_afr_vs_sas', 'FST_afr_vs_fin', 'FST_afr_vs_asj', 'FST_afr_vs_eas', 'FST_amr_vs_nfe', 'FST_amr_vs_sas', 'FST_amr_vs_fin', 'FST_amr_vs_asj', 'FST_amr_vs_eas', 'FST_nfe_vs_sas', 'FST_nfe_vs_fin', 'FST_nfe_vs_asj', 'FST_nfe_vs_eas', 'FST_sas_vs_fin', 'FST_sas_vs_asj', 'FST_sas_vs_eas', 'FST_fin_vs_asj', 'FST_fin_vs_eas', 'FST_asj_vs_eas']


## 3. Sample VCF File

The file `chr21_sample_100k` contains the first 100,000 variants from chromosome 21 of the gnomAD v4 exome and is provided directly in this repository.

💡 To run on your own chromosome file, download the gnomAD v4 exome VCF from https://gnomad.broadinstitute.org/downloads#v4, place it in the same directory, and replace `chr21_sample_100k` below with your filename.

In [5]:
import time

# Ensure output directory exists
os.makedirs('../output', exist_ok=True)

# Input VCF provided in this repository
input_vcf = '../data/pipeline/chr21_sample_100k'

# All generated files go to output/
name = '../output/chr21_sample_100k'
nameCSV = name + '.csv'

# Count variants in the sample file
print(f'Using pre-provided sample VCF: {input_vcf}')
variant_count = 0
with open(input_vcf, 'r') as f:
    for line in f:
        if not line.startswith('#'):
            variant_count += 1
n_variants = variant_count

print(f'Sample contains {n_variants:,} variants')
print(f'Output files will be written to: output/')
print('💡 For production use, replace input_vcf with your own VCF file path')

Using pre-provided sample VCF: ../data/pipeline/chr21_sample_100k
Sample contains 100,000 variants
Output files will be written to: output/
💡 For production use, replace input_vcf with your own VCF file path


## 4. Run VCFtoCSV_Exome (Main Conversion Function)

This is the core function that does all the heavy lifting - VCF parsing, annotation, HWE calculation, etc.

In [6]:
print("Starting VCFtoCSV_Exome conversion...")
print("This function performs:")
print("- VCF parsing and basic variant extraction")
print("- Population allele count extraction")
print("- Hardy-Weinberg equilibrium p-value calculation")
print("- VEP consequence annotation")
print("- External database matching (AGAIN, BPHunter, HGMD, UM etc)")
print()

# Track processing time and add progress monitoring
import time
import threading
import os

def monitor_progress(input_file, output_file, stop_event, expected_variants):
    """Monitor processing progress by checking output file"""
    start_time = time.time()
    
    while not stop_event.is_set():
        if os.path.exists(output_file):
            try:
                # Count lines in output CSV (minus header)
                with open(output_file, 'r') as f:
                    lines = sum(1 for _ in f) - 1  # Subtract header
                
                elapsed = time.time() - start_time
                progress = (lines / expected_variants) * 100 if expected_variants > 0 else 0
                variants_per_sec = lines / elapsed if elapsed > 0 else 0
                
                print(f"\r⏱️  Processed: {lines:,}/{expected_variants:,} variants ({progress:.1f}%) | "
                      f"Rate: {variants_per_sec:.0f} variants/sec | "
                      f"Elapsed: {elapsed:.0f}s", end="", flush=True)
            except Exception:
                # Ignore errors during monitoring (e.g., file being written)
                pass
        
        time.sleep(5)  # Update every 5 seconds

# Start progress monitoring
print(f"🚀 Processing {n_variants:,} variants...")
stop_monitoring = threading.Event()
monitor_thread = threading.Thread(target=monitor_progress, 
                                   args=(input_vcf, nameCSV, stop_monitoring, n_variants))
monitor_thread.start()

start_time = time.time()

# Call the exact function from the original pipeline
VCFtoCSV_Exome(input_vcf, medList, quantityList, ethnList, sexList, p_valueList, nameCSV, 1,
               listAGAIN, listBPHUNTER, listHGMD, listpLOF, FSTs_header, Merged_Intervals_37)

# Stop monitoring
stop_monitoring.set()
monitor_thread.join()

end_time = time.time()
elapsed_time = end_time - start_time

print(f"\n\n✅ VCFtoCSV_Exome completed! Output: {nameCSV}")
print(f"⏱️  Processing time: {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")

# Load and inspect the result
df_raw = pd.read_csv(nameCSV)
print(f"\nGenerated CSV shape: {df_raw.shape}")
print(f"Columns: {len(df_raw.columns)}")
print(f"Processing rate: {len(df_raw)/elapsed_time:.0f} variants/second")
print("\nFirst few columns:")
print(df_raw.columns[:20].tolist())

Starting VCFtoCSV_Exome conversion...
This function performs:
- VCF parsing and basic variant extraction
- Population allele count extraction
- Hardy-Weinberg equilibrium p-value calculation
- VEP consequence annotation
- External database matching (AGAIN, BPHunter, HGMD, UM etc)

🚀 Processing 100,000 variants...
⏱️  Processed: 100,000/100,000 variants (100.0%) | Rate: 282931 variants/sec | Elapsed: 0s

/Users/khorenponsin/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/khorenponsin/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/khorenponsin/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.py:269: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/khorenponsin/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/khorenponsin/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.py:261: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


⏱️  Processed: 93,429/100,000 variants (93.4%) | Rate: 1088 variants/sec | Elapsed: 86s7s

✅ VCFtoCSV_Exome completed! Output: ../output/chr21_sample_100k.csv
⏱️  Processing time: 90.9 seconds (1.5 minutes)

Generated CSV shape: (100000, 108)
Columns: 108
Processing rate: 1100 variants/second

First few columns:
['chromo', 'position', 'ID', 'ref', 'alt', 'qual', 'filter', 'n_alt_alleles', 'AC_total', 'AN_total', 'AF_total', 'nhomalt_total', 'AC_afr', 'AN_afr', 'AF_afr', 'nhomalt_afr', 'AC_amr', 'AN_amr', 'AF_amr', 'nhomalt_amr']


/var/folders/q8/5rgt92sx7b74krhrt6l4_7hw0000gn/T/ipykernel_36527/4118104536.py:63: DtypeWarning: Columns (70,105) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(nameCSV)


## 5. Apply Filter_filter (Quality Filtering)

Apply the initial quality filters:

In [7]:
print("Applying Filter_filter...")
print("This applies basic quality filters to the dataset")

# Monitor filtering progress
filter_start_time = time.time()
filter_input = name + '.csv'
filter_output = name + '_filter.csv'

print(f"🔍 Filtering {len(df_raw):,} variants...")

# Start progress monitoring for filtering
stop_filter_monitoring = threading.Event()
filter_monitor_thread = threading.Thread(target=monitor_progress, 
                                         args=(filter_input, filter_output, stop_filter_monitoring, len(df_raw)))
filter_monitor_thread.start()

# Apply the filter function (exact copy from pipeline)
Filter_filter(filter_input, filter_output)

# Stop filtering monitoring
stop_filter_monitoring.set()
filter_monitor_thread.join()

filter_end_time = time.time()
filter_elapsed = filter_end_time - filter_start_time

print(f"\n\n✅ Filtering completed! Output: {filter_output}")
print(f"⏱️  Filtering time: {filter_elapsed:.1f} seconds")

# Compare before and after filtering
df_filtered = pd.read_csv(filter_output)
print(f"\nBefore filtering: {len(df_raw):,} variants")
print(f"After filtering: {len(df_filtered):,} variants")
print(f"Filtered out: {len(df_raw) - len(df_filtered):,} variants ({(len(df_raw) - len(df_filtered))/len(df_raw)*100:.1f}%)")
print(f"Filtering rate: {len(df_raw)/filter_elapsed:.0f} variants/second")

print(f"\nFiltered dataset columns ({len(df_filtered.columns)} total):")
print("First 10 columns:", df_filtered.columns[:10].tolist())

# Find the correct column names for total population
ac_col = 'AC_total' if 'AC_total' in df_filtered.columns else 'AC'
af_col = 'AF_total' if 'AF_total' in df_filtered.columns else 'AF'
pv_col = 'pv_total' if 'pv_total' in df_filtered.columns else None

print("\nExample of filtered data:")
# Display basic variant info columns
display_cols = ['chromo', 'position', 'ref', 'alt']

# Add available population data columns
if ac_col in df_filtered.columns:
    display_cols.append(ac_col)
if af_col in df_filtered.columns:
    display_cols.append(af_col)
if pv_col and pv_col in df_filtered.columns:
    display_cols.append(pv_col)

print(df_filtered[display_cols].head())

Applying Filter_filter...
This applies basic quality filters to the dataset
🔍 Filtering 100,000 variants...
⏱️  Processed: 8,030/100,000 variants (8.0%) | Rate: 295609 variants/sec | Elapsed: 0s

✅ Filtering completed! Output: ../output/chr21_sample_100k_filter.csv
⏱️  Filtering time: 5.0 seconds

Before filtering: 100,000 variants
After filtering: 8,030 variants
Filtered out: 91,970 variants (92.0%)
Filtering rate: 19865 variants/second

Filtered dataset columns (108 total):
First 10 columns: ['chromo', 'position', 'ID', 'ref', 'alt', 'qual', 'filter', 'n_alt_alleles', 'AC_total', 'AN_total']

Example of filtered data:
  chromo  position ref alt  AC_total  AF_total  pv_total
0  chr21   5031911   G   C         1  0.000002  0.500000
1  chr21   5031914   G   T         2  0.000004  0.500001
2  chr21   5031916   T   C         1  0.000002  0.500000
3  chr21   5031924   C   T        36  0.000069  0.500603
4  chr21   5031933   C   G         1  0.000002  0.500000


## 6. Create Population-Specific CSV Files

Use csvEachPop_Exome to generate separate CSV files for each population:

In [8]:
print("Creating population-specific CSV files...")
print(f"Generating separate datasets for {len(ethnList)} populations")

# Call the exact function from the pipeline
csvEachPop_Exome(name + '_filter.csv', medList, quantityList, ethnList, sexList, p_valueList, name + '_filter_')

print("\nPopulation-specific CSV files created!")

# Check the created files and their sizes
pop_file_info = []
for pop in ethnList:
    pop_file = f"{name}_filter_{pop}.csv"
    if os.path.exists(pop_file):
        pop_df = pd.read_csv(pop_file)
        pop_file_info.append({
            'Population': pop,
            'File': pop_file,
            'Variants': len(pop_df),
            'Sample_Size': nbAllesPerEthn[ethnList.index(pop)] // 2
        })

pop_summary = pd.DataFrame(pop_file_info)
print("\nPopulation file summary:")
print(pop_summary.to_string(index=False))

Creating population-specific CSV files...
Generating separate datasets for 8 populations

Population-specific CSV files created!

Population file summary:
Population                                         File  Variants  Sample_Size
     total ../output/chr21_sample_100k_filter_total.csv      8030       729178
       afr   ../output/chr21_sample_100k_filter_afr.csv      8030        16677
       amr   ../output/chr21_sample_100k_filter_amr.csv      8030        22359
       nfe   ../output/chr21_sample_100k_filter_nfe.csv      8030       554421
       sas   ../output/chr21_sample_100k_filter_sas.csv      8030        43107
       fin   ../output/chr21_sample_100k_filter_fin.csv      8030        26704
       asj   ../output/chr21_sample_100k_filter_asj.csv      8030        13057
       eas   ../output/chr21_sample_100k_filter_eas.csv      8030        19834


## 7. Apply Population-Specific Filters

For each population, apply the same filters as in the original pipeline:

In [9]:
print("Applying population-specific filters...")
print("For each population:")
print("- Filter_BI: Biallelic sites filter")
print("- Filter_Auto_AC_CallRate: Allele count and call rate filters")
print("- add_MORE_Exome: Additional annotations")
print()

final_stats = []
pop_processing_start = time.time()

for i in range(len(ethnList)):
    pop = ethnList[i]
    pop_file = name + '_filter_' + pop + '.csv'
    
    if os.path.exists(pop_file):
        print(f"🔄 Processing {pop} population ({i+1}/{len(ethnList)})...")
        pop_start_time = time.time()
        
        # Get initial count
        df_before = pd.read_csv(pop_file)
        initial_count = len(df_before)
        print(f"   📊 Starting with {initial_count:,} variants")
        
        # Apply filters (exact copy from pipeline)
        print(f"   🔍 Applying Filter_BI...")
        Filter_BI(pop_file, pop_file)
        
        print(f"   🔍 Applying Filter_Auto_AC_CallRate...")
        Filter_Auto_AC_CallRate(pop_file, 5, 0.9, nbAllesPerEthn[i], pop_file)
        
        print(f"   🔍 Applying add_MORE_Exome...")
        add_MORE_Exome(pop_file, pop_file)
        
        # Get final count
        df_after = pd.read_csv(pop_file)
        final_count = len(df_after)
        
        pop_elapsed = time.time() - pop_start_time
        
        final_stats.append({
            'Population': pop,
            'Initial_Variants': initial_count,
            'Final_Variants': final_count,
            'Filtered_Out': initial_count - final_count,
            'Retention_Rate': f"{final_count/initial_count*100:.1f}%" if initial_count > 0 else "0%",
            'Processing_Time': f"{pop_elapsed:.1f}s"
        })
        
        print(f"   ✅ {pop}: {initial_count:,} → {final_count:,} variants "
              f"({final_count/initial_count*100:.1f}% retained) in {pop_elapsed:.1f}s")
        print()

total_pop_elapsed = time.time() - pop_processing_start
print("🎉 Population-specific filtering completed!")
print(f"⏱️  Total processing time: {total_pop_elapsed:.1f} seconds ({total_pop_elapsed/60:.1f} minutes)")

# Summary table
final_summary = pd.DataFrame(final_stats)
print("\nFinal filtering summary:")
print(final_summary.to_string(index=False))

Applying population-specific filters...
For each population:
- Filter_BI: Biallelic sites filter
- Filter_Auto_AC_CallRate: Allele count and call rate filters
- add_MORE_Exome: Additional annotations

🔄 Processing total population (1/8)...
   📊 Starting with 8,030 variants
   🔍 Applying Filter_BI...
   🔍 Applying Filter_Auto_AC_CallRate...
   🔍 Applying add_MORE_Exome...
   ✅ total: 8,030 → 706 variants (8.8% retained) in 0.4s

🔄 Processing afr population (2/8)...
   📊 Starting with 8,030 variants
   🔍 Applying Filter_BI...
   🔍 Applying Filter_Auto_AC_CallRate...
   🔍 Applying add_MORE_Exome...
   ✅ afr: 8,030 → 140 variants (1.7% retained) in 0.4s

🔄 Processing amr population (3/8)...
   📊 Starting with 8,030 variants
   🔍 Applying Filter_BI...
   🔍 Applying Filter_Auto_AC_CallRate...
   🔍 Applying add_MORE_Exome...
   ✅ amr: 8,030 → 122 variants (1.5% retained) in 0.4s

🔄 Processing nfe population (4/8)...
   📊 Starting with 8,030 variants
   🔍 Applying Filter_BI...
   🔍 Applying Fi

## 8. Apply Hardy-Weinberg Equilibrium Corrections (addCE)

Apply population-specific HWE corrections using calibrated alpha parameters. This step corresponds to the `addCE.py` script.

### Important Note on CHWE Parameterization

In our academic, we describe **CHWE (Corrected Hardy-Weinberg Equilibrium)** as:
- CHWE = **2γp(1-p)** where γ (gamma) is the correction parameter

However, here, the formula is expressed as:
- CHWE = **αp(1-p)** where α (alpha) is the correction parameter

**Therefore**: **α = 2γ** 

In [10]:
# Import HWE correction functions (exact copy from addCE.py)
from HWE import LH_Distrib, midPvalue, nbHete, ExpectedHete, ExpecteHomAlt, CE_Distrib, degreeOfDiseq, ExpectedHete_CE
from CorrectingHWE_ExomeV4 import addCE_ExomeV4

print("Applying Hardy-Weinberg Equilibrium corrections...")
print("This step applies population-specific corrections using calibrated alpha parameters.")
print()

# Define populations and alpha parameters (exact copy from addCE.py)
listE = ['afr', 'amr', 'nfe', 'sas', 'fin', 'asj', 'eas']
listAlpha = [1.9874139849098698, 1.965243530926951, 1.9970521767231806,
             1.9533794840984369, 1.9945152397148354, 1.9978419115917891,
             1.9856708299528565]

print(f"Processing {len(listE)} populations with calibrated alpha parameters:")
for i in range(len(listE)):
    print(f"   {listE[i]}: α = {listAlpha[i]:.4f}")

print()

ce_start_time = time.time()
ce_stats = []

# Apply HWE corrections to each population (exact copy from addCE.py logic)
for i in range(len(listE)):
    pop = listE[i]
    alpha = listAlpha[i]
    input_file = name + '_filter_' + pop + '.csv'
    output_file = name + '_filter_' + pop + '_CE.csv'

    if os.path.exists(input_file):
        print(f"🔧 Applying HWE correction to {pop} (α = {alpha:.4f})...")

        # Get input size
        df_before = pd.read_csv(input_file)
        input_variants = len(df_before)

        correction_start = time.time()

        # Apply the correction function (exact copy from addCE.py)
        addCE_ExomeV4(input_file, alpha, output_file)

        correction_elapsed = time.time() - correction_start

        # Get output size
        df_after = pd.read_csv(output_file)
        output_variants = len(df_after)

        ce_stats.append({
            'Population': pop,
            'Alpha_Parameter': f"{alpha:.4f}",
            'Input_Variants': input_variants,
            'Output_Variants': output_variants,
            'Processing_Time': f"{correction_elapsed:.1f}s"
        })

        print(f"   ✅ {pop}: {input_variants:,} variants processed → {output_file}")
        print(f"      ⏱️  Processing time: {correction_elapsed:.1f}s")
        print()
    else:
        print(f"   ⚠️  Skipping {pop}: input file {input_file} not found")
        print()

total_ce_elapsed = time.time() - ce_start_time

print("🎉 Hardy-Weinberg Equilibrium corrections completed!")
print(f"⏱️  Total HWE correction time: {total_ce_elapsed:.1f} seconds")

# Summary table
ce_summary = pd.DataFrame(ce_stats)
if not ce_summary.empty:
    print("\nHWE correction summary:")
    print(ce_summary.to_string(index=False))

print(f"\n📁 Generated files with '_CE.csv' suffix:")
for i in range(len(listE)):
    pop = listE[i]
    ce_file = name + '_filter_' + pop + '_CE.csv'
    if os.path.exists(ce_file):
        print(f"   ✅ {ce_file}")
    else:
        print(f"   ❌ {ce_file} (not created)")


Applying Hardy-Weinberg Equilibrium corrections...
This step applies population-specific corrections using calibrated alpha parameters.

Processing 7 populations with calibrated alpha parameters:
   afr: α = 1.9874
   amr: α = 1.9652
   nfe: α = 1.9971
   sas: α = 1.9534
   fin: α = 1.9945
   asj: α = 1.9978
   eas: α = 1.9857

🔧 Applying HWE correction to afr (α = 1.9874)...
   ✅ afr: 140 variants processed → ../output/chr21_sample_100k_filter_afr_CE.csv
      ⏱️  Processing time: 0.1s

🔧 Applying HWE correction to amr (α = 1.9652)...
   ✅ amr: 122 variants processed → ../output/chr21_sample_100k_filter_amr_CE.csv
      ⏱️  Processing time: 0.1s

🔧 Applying HWE correction to nfe (α = 1.9971)...
   ✅ nfe: 498 variants processed → ../output/chr21_sample_100k_filter_nfe_CE.csv
      ⏱️  Processing time: 3.1s

🔧 Applying HWE correction to sas (α = 1.9534)...
   ✅ sas: 191 variants processed → ../output/chr21_sample_100k_filter_sas_CE.csv
      ⏱️  Processing time: 0.3s

🔧 Applying HWE cor